In [ ]:
import os # For file handling and directory management
import cv2 # OpenCV for image processing
import numpy as np # For numerical operations and array handling
import pandas as pd # For data manipulation and results logging
import matplotlib.pyplot as plt # For visualizing results and data distributions
import seaborn as sns # For enhanced data visualization
from sklearn.model_selection import train_test_split # For splitting data into training, validation, and test sets
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score # For evaluating model performance

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded. Ready to start.")

Libraries loaded. Ready to start.


In [ ]:
def apply_filter(image, filter_type): # Apply filters to the image based on the specified filter type
   
    # 1. RAW (No processing, just the original image)
    if filter_type == "Raw":
        return image 
    
    # 2. Gaussian Filter (Smoothing)
    elif filter_type == "Gaussian":
        return cv2.GaussianBlur(image, (5, 5), 0)
    
    # 3. Median Filter (Noise Removal)
    elif filter_type == "Median":
        return cv2.medianBlur(image, 5)
    
    # 4. Bilateral Filter (Edge Preserving Smoothing)
    elif filter_type == "Bilateral":
        return cv2.bilateralFilter(image, 9, 75, 75)
    
    # 5. Combined (Sequential Application)
    elif filter_type == "Combined":
        img = cv2.GaussianBlur(image, (5, 5), 0)
        img = cv2.medianBlur(img, 5)
        return cv2.bilateralFilter(img, 9, 75, 75)
    
    return image

In [ ]:
BASE_DIR = "Dataset"
IMG_SIZE = 64  # Resizing to 64x64 pixels for efficiency

def load_data(total_needed, filter_name): # Load images, apply filters, and prepare data for ML models
    print(f"\n--- Loading {total_needed} images ({filter_name}) ---")
    data = []
    labels = []
    
    
    per_class = total_needed // 2
    
    categories = ["no", "yes"] # 0 = No, 1 = Yes
    
    for label_code, category in enumerate(categories):
        folder_path = os.path.join(BASE_DIR, category)
        
        # Check if folder exists
        if not os.path.exists(folder_path):
            print(f"Error: Folder '{folder_path}' not found!")
            return np.array([]), np.array([])
            
        # Get list of image files
        files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        # ERROR CHECK
        if len(files) < per_class:
            print(f"CRITICAL ERROR: You asked for {total_needed} images total ({per_class} per class),")
            print(f"but folder '{category}' only has {len(files)} images.")
            return np.array([]), np.array([])

        # Take only the first N images needed
        selected_files = files[:per_class]
        
        for file in selected_files:
            img_path = os.path.join(folder_path, file)
            img = cv2.imread(img_path, 0) # Load as Grayscale
            
            if img is not None:
                # Resize and Filter
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = apply_filter(img, filter_name)
                
                data.append(img.flatten()) # Flatten to 1D array
                labels.append(label_code)
                
    return np.array(data), np.array(labels)

In [ ]:


# Setup
dataset_sizes = [100, 500]
filters = ["Raw", "Gaussian", "Median", "Bilateral", "Combined"]
results_log = []  # This list will hold all our data

# Define Models
models = {
    "KNN": KNeighborsClassifier(n_neighbors=3),
    "SVM": SVC(kernel='linear', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

#  EXECUTION 
print("Running Experiments... Please wait.")
print(f"{'SIZE':<5} | {'FILTER':<10} | {'MODEL':<15} | {'ACC':<6} | {'PREC':<6} | {'REC':<6} | {'F1':<6}")
print("-" * 80)

for size in dataset_sizes:
    for filter_type in filters:
        
        # 1. Load the data
        X, y = load_data(size, filter_type)
        if len(X) == 0: break
            
        # 2. Loop through each model
        for model_name, model in models.items():
            
            # Storage for the 5 runs
            model_accs = []
            model_precs = []
            model_recs = []
            model_f1s = []
            
            # 3. REPEAT THE SPLIT 5 TIMES (for reliability)
            for i in range(5):
                # SPLIT A: 70% Train, 30% Temp
                X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=i, stratify=y)
                
                # SPLIT B: Split that 30% Temp into 15% Val / 15% Test
                X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=i, stratify=y_temp)
                
                # Train
                model.fit(X_train, y_train)
                
                # Predict (on Test Set)
                y_pred = model.predict(X_test)
                
                # Record Scores
                model_accs.append(accuracy_score(y_test, y_pred))
                model_precs.append(precision_score(y_test, y_pred, zero_division=0))
                model_recs.append(recall_score(y_test, y_pred, zero_division=0))
                model_f1s.append(f1_score(y_test, y_pred, zero_division=0))
            
            # 4. Calculate Averages
            avg_acc = np.mean(model_accs)
            avg_prec = np.mean(model_precs)
            avg_rec = np.mean(model_recs)
            avg_f1 = np.mean(model_f1s)
            
            # Print Progress Row 
            print(f"{size:<5} | {filter_type:<10} | {model_name:<15} | {avg_acc:.2f}   | {avg_prec:.2f}   | {avg_rec:.2f}   | {avg_f1:.2f}")
            
            # Log for the Final Table
            results_log.append({
                "Dataset Size": size,
                "Filter Type": filter_type,
                "Model": model_name,
                "Accuracy": avg_acc,
                "Precision": avg_prec,
                "Recall": avg_rec,
                "F1 Score": avg_f1
            })

#  FINAL TABLE GENERATION 
print("\n" + "="*30)
print(" FINAL RESULTS TABLE ")
print("="*30)

# Create DataFrame
df_results = pd.DataFrame(results_log)

# formatting
pd.options.display.float_format = '{:,.2f}'.format

# Display the full table
display(df_results)

In [ ]:
# Cell: CNN Experiment on 100 and 500 Datasets (Raw Data Only)

import numpy as np # For numerical operations and array handling
import pandas as pd # For data manipulation and results logging
import tensorflow as tf # For building and training the CNN
from tensorflow.keras import layers, models # For defining the CNN architecture
from sklearn.model_selection import train_test_split # For splitting data into training and test sets
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score # For evaluating model performance

# Setup
dataset_sizes = [100, 500]
cnn_results_log = []


def build_cnn():
    """Builds a fresh, lightweight CNN for each cross-validation fold."""
    model = models.Sequential([
        # First Convolutional Block
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 1)),
        layers.MaxPooling2D((2, 2)),
        
        # Second Convolutional Block
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Flatten to 1D for the final decision
        layers.Flatten(),
        
        # Dense Layers with Dropout to prevent overfitting on small data
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5), # Drops 50% of connections to force robust learning
        layers.Dense(1, activation='sigmoid') # Sigmoid for Binary Classification (0 to 1)
    ])
    
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model


print("Running CNN Experiments on RAW data... This will take a moment.")
print(f"{'SIZE':<5} | {'ACC':<6} | {'PREC':<6} | {'REC':<6} | {'F1':<6}")
print("-" * 50)

for size in dataset_sizes:
        
    # 1. Load the RAW data
    # (Assuming load_data still needs the filter argument string to fetch the correct baseline)
    X_flat, y = load_data(size, "Raw")
    if len(X_flat) == 0: break
        
    # 2. PREPARE DATA FOR THE CNN
    # Reshape from (Samples, 4096) to (Samples, 64, 64, 1)
    X = X_flat.reshape(-1, 64, 64, 1)
    
    # NORMALIZE PIXELS: Scale from 0-255 to 0.0-1.0 (Crucial for CNNs)
    X = X / 255.0
    
    # Storage for the 5 runs
    fold_accs, fold_precs, fold_recs, fold_f1s = [], [], [], []
    
    # 3. 5-FOLD CROSS VALIDATION
    for i in range(5):
        # SPLIT: 80% Train, 20% Test (Simplified for this small data CNN test)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=i, stratify=y)
        
        # Build a fresh model for this fold
        cnn_model = build_cnn()
        
        # Train the CNN (Silent mode verbose=0 to keep output clean)
        cnn_model.fit(X_train, y_train, epochs=15, batch_size=16, verbose=0)
        
        # Predict (Outputs probabilities, so we round to 0 or 1)
        y_pred_probs = cnn_model.predict(X_test, verbose=0)
        y_pred = np.round(y_pred_probs).flatten()
        
        # Record Scores
        fold_accs.append(accuracy_score(y_test, y_pred))
        fold_precs.append(precision_score(y_test, y_pred, zero_division=0))
        fold_recs.append(recall_score(y_test, y_pred, zero_division=0))
        fold_f1s.append(f1_score(y_test, y_pred, zero_division=0))
    
    # 4. Calculate Averages
    avg_acc = np.mean(fold_accs)
    avg_prec = np.mean(fold_precs)
    avg_rec = np.mean(fold_recs)
    avg_f1 = np.mean(fold_f1s)
    
    # Print Row
    print(f"{size:<5} | {avg_acc:.2f}   | {avg_prec:.2f}   | {avg_rec:.2f}   | {avg_f1:.2f}")
    
    cnn_results_log.append({
        "Dataset Size": size,
        "Filter Type": "Raw (Normalized)",
        "Model": "CNN",
        "Accuracy": avg_acc,
        "Precision": avg_prec,
        "Recall": avg_rec,
        "F1 Score": avg_f1
    })

# Show final table
display(pd.DataFrame(cnn_results_log))

In [ ]:


# Convert results to DataFrame
df = pd.DataFrame(results_log)

# 1. Show the best performing combinations
print("\nTop 5 Performing Models (by Accuracy):")
if not df.empty:
    display(df.sort_values(by="Accuracy", ascending=False).head(5))
else:
    print("No results to display. Check if Cell 4 ran correctly.")

# 2. Bar Chart: Compare Filters on the 500 Image Dataset

if 500 in df["Dataset Size"].values:
    plt.figure(figsize=(14, 6))
    subset = df[df["Dataset Size"] == 500]
    
    
    sns.barplot(data=subset, x="Filter Type", y="Accuracy", hue="Model", palette="viridis", errorbar=None)
    plt.title("Effect of Filters on Accuracy (500 Images)")
    plt.ylim(0, 1.1)
    plt.ylabel("Test Accuracy")
    plt.grid(axis='y', alpha=0.3)
    plt.show()

# 3. Line Chart: 100 vs 500 Images Performance
if not df.empty:
    plt.figure(figsize=(10, 5))
    
    sns.pointplot(data=df, x="Dataset Size", y="F1 Score", hue="Model", errorbar=None)
    plt.title("Impact of Dataset Size on F1 Score")
    plt.ylabel("F1 Score")
    plt.grid(True)
    plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay # For generating and displaying the confusion matrix


print("Generating Confusion Matrix for SVM (Raw - 500 Images)...")

# Reload specific data
X, y = load_data(500, "Raw")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train single instance
model = SVC(kernel='linear', random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Plot
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Tumor", "Tumor"])
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix: Where did the model fail?")
plt.show()